# Needs Assessment

**Project:** Acled Events Geoprocessing

**Notebook:** `acled_events_geoprocessing.ipynb`

**Author:** Yuseof J

**Created:** 15/11/2026 | **Last updated:** 15/11/2026

## Purpose
   This file is meant to calculate aggregate metrics of violence by governorate. However, data is not granular enough, see notes below 

## Inputs

- `data/raw/boundaries/`: HDX admin boundaries (COD-AB)

## Outputs
- `data/processed/[filename].csv`: what it contains

## Notes and caveats
    - Data from ACLED is aggregated at the admin1 level, so I won't be able to perform this analysis at the governorate level as originally intended. Currently awaiting access from ACLED pending a request, but not sure I will be able to get it. As such, this may have to exist only at the admin1 level as a POC, with explicit outline of data limitations and future plans given more granular access

In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CRS = 'EPSG:4326'

### Data imports / exports

In [ ]:
# import paths
PATH_TO_ACLED_EVENTS = '../data/raw/Middle-East_aggregated_data_up_to_week_of-2026-10-31.csv'
PATH_TO_ADMIN_BOUNDARIES = '../data/raw/pse_admin_boundaries.shp/pse_admin2.shp'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = '../data/processed/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = '../data/processed/acled/conflict_events_geolocated.csv'

# import data
df_events = pd.read_csv(PATH_TO_ACLED_EVENTS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

In [ ]:
df_events.head()

In [ ]:
print(f"{df_events.CENTROID_LATITUDE.nunique()} total lat, {df_events.CENTROID_LONGITUDE.nunique()} total long")

In [ ]:
gdf_boundaries.head()

In [ ]:
print(f"{gdf_boundaries.center_lat.nunique()} total lat, {gdf_boundaries.center_lon.nunique()} total long")

### Preprocessing

In [ ]:
# strip whitespaces
string_cols = ['REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'DISORDER_TYPE']
df_events[string_cols] = df_events[string_cols].apply(lambda x: x.str.strip())

# format date columns
df_events['WEEK'] = pd.to_datetime(df_events['WEEK'])

# create date grouping columns
df_events['YEAR'] = df_events['WEEK'].dt.year
df_events['YEARMO'] = df_events['WEEK'].dt.to_period('M').astype(str)
df_events['YEAR_WEEK'] = df_events['WEEK'].dt.to_period('W').astype(str)

mask_palestine_events = df_events.COUNTRY == 'Palestine'
df_events = df_events[mask_palestine_events]

### Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_events = gpd.GeoDataFrame(
    df_events,
    geometry=gpd.points_from_xy(
        df_events['CENTROID_LONGITUDE'],
        df_events['CENTROID_LATITUDE']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_events.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_events = gdf_events.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_events.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_events_joined = gpd.sjoin(
    gdf_events,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
print("Total events:", len(gdf_events_joined))
print("Matched events:", gdf_events_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_events_joined['adm1_name'].isna().sum())

In [ ]:
unmatched = gdf_events_joined[gdf_events_joined['adm1_name'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

In [ ]:
 gdf_events_joined['adm1_name'].value_counts()

In [ ]:
 gdf_events_joined['adm2_name'].value_counts()

In [ ]:
# do all admin 1 names agree between sources?
assert(len(gdf_events_joined[gdf_events_joined.ADMIN1 != gdf_events_joined.adm1_name]) == 0)

In [ ]:
# Check whether any original event IDs appear more than once
print(gdf_events_joined.index.value_counts().value_counts().sort_index())

<br>

### Governorate Aggregates

#### Total Events (all)

In [ ]:
#gdf_events_joined.head()

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

#### Total Events (by type)

Across all years

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'EVENT_TYPE'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

<br>

Broken down by year, and ratio (normalized count) is for the year

This would allow us to see how specific event types are changing over time by governorate

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'YEAR', 'EVENT_TYPE'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name', 'YEAR'])['count'].transform('sum')
).round(2)

event_counts
# print(event_counts.to_string()) # to see all rows

<br>

Broken down by event subtype (could also be interesting to see subtype broken down by year, but would be too large to comprehend upon manual visual inspection)

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'SUB_EVENT_TYPE'])['EVENTS']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

<br>

#### Total Fatalities

Across all years and event types

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name'])['FATALITIES']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

Across all years, broken down by event type

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'EVENT_TYPE'])['FATALITIES']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name'])['count'].transform('sum')
).round(2)

event_counts

Broken down by year, and ratio (normalized count) is for the year

This would allow us to see how specific event types are changing over time

In [ ]:
event_counts = (
    gdf_events_joined
    .groupby(['adm1_name', 'adm2_name', 'YEAR', 'EVENT_TYPE'])['FATALITIES']
    .sum()
    .reset_index(name='count')
)

event_counts['normalized_count'] = (
    event_counts['count'] /
    event_counts.groupby(['adm1_name', 'adm2_name', 'YEAR'])['count'].transform('sum')
).round(2)

event_counts
#print(event_counts.to_string()) # to see all rows

#### Rate of Change

# TODO: PAUSING HERE (11/16)

Data from ACLED is aggregated at the admin1 level, so I won't be able to perform this analysis at the governorate level as originally intended. Currently awaiting access from ACLED pending a request, but not sure I will be able to get it. As such,
this may have to exist only at the admin1 level as a POC, with explicit outline of data limitations and future plans given more granular access

- rate of change of violence - last week, last 2 weeks, last month, last 3 months, last 6 months, last year

In [ ]:
gdf_events_joined.head()


In [ ]:
gdf_events_joined.CENTROID_LATITUDE.unique()

# rate of change of mob violence per gov

left off here: workshopping this for Claude: 

I want to calculate rate of change of violence (i.e. num of violent events), with the option to do it by all event types, or by specific event types. I want to do this over a two week period, 
a 1 month period, 3 month period, 6 months, and 1 year. By rate of change, I mean I want to know the slope of the line if I were to plot number of events per location on a line chart over the specified time window (unless you see a better way to do this, but if so, explain it to me).
The relevant columns are:

"geometry": point of violence

"adm2_name": governorate

"YEAR_WEEK": which is in the form 2016-04-04/2016-04-10

"SUB_EVENT_TYPE": I want to be able to filter by this, or else select all sub event types

"EVENTS": this is the number of instances of this event type that showed up at that geometry

I want two different granularities:

    - rate per point (group by "geometry", slope of "events" over specified timeframe - make sure to flag as NA if window is not long enough (i.e. violence in that areea started in last week and we're calculating 2 week window))
    
    - rate per governorate (group by "adm2_name", sum "events" column across all points in the admin, slope of this sum across time window)

Code below needs some QA checks (ensure the calculated slopes are aligned with the trends seen in the charts of events over time in acled_events_eda notebook)

In [ ]:
"""
Rate of change (slope) of weekly violent-event counts, per point and per governorate.

Expected columns in the input GeoDataFrame:
    geometry        point location of the events
    adm2_name       governorate
    YEAR_WEEK       e.g. "2016-04-04/2016-04-10"
    SUB_EVENT_TYPE  filterable event type
    EVENTS          number of events of that type at that point that week

Slope units: change in events per week, per week (OLS slope over weekly counts).
"""
import numpy as np
import pandas as pd

# Window lengths in weeks. Edit to taste (1m ~ 4 weeks, 3m ~ 13, 6m ~ 26, 1y ~ 52).
WINDOWS = {"2w": 2, "1m": 4, "3m": 12, "6m": 24, "1y": 52}


def prepare(gdf, sub_event_types=None):
    """Optionally filter by SUB_EVENT_TYPE, parse week start dates, build a point key."""
    d = gdf.copy()
    if sub_event_types:
        d = d[d["SUB_EVENT_TYPE"].isin(sub_event_types)]
    d = d[d["EVENTS"] > 0]
    d["week_start"] = pd.to_datetime(d["YEAR_WEEK"].str.split("/").str[0])
    d["point_id"] = d.geometry.to_wkt()
    return d


def rate_of_change(d, key, windows=WINDOWS, as_of=None):
    """
    Slope of weekly event counts for each value of `key` ("point_id" or "adm2_name").

    - Weeks with no rows are treated as zero events (the data only records weeks
      where something happened), so the series is zero-filled over the window.
    - If a key's first-ever event falls inside the window (i.e. we haven't observed
      it for the full window), slope is NaN and status says why.
    - as_of is the week-start date of the last week to include. Defaults to the
      latest week in the data. If the latest week is incomplete, pass the previous
      week start instead.
    """
    as_of = pd.Timestamp(as_of) if as_of is not None else d["week_start"].max()
    d = d[d["week_start"] <= as_of]

    weekly = d.groupby([key, "week_start"])["EVENTS"].sum().unstack(fill_value=0)
    # Pad the timeline back far enough for the longest window, even if the (possibly
    # filtered) data starts later. Padding weeks are zeros; first_seen decides NA.
    max_w = max(windows.values())
    start = min(d["week_start"].min(), as_of - pd.Timedelta(weeks=max_w - 1))
    all_weeks = pd.date_range(start, as_of, freq="7D")
    weekly = weekly.reindex(columns=all_weeks, fill_value=0)

    first_seen = d.groupby(key)["week_start"].min().reindex(weekly.index)

    out = []
    for label, w in windows.items():
        win = weekly.iloc[:, -w:]
        window_start = win.columns[0]
        y = win.to_numpy(dtype=float)

        x = np.arange(w) - (w - 1) / 2          # centered week index
        slope = ((y - y.mean(axis=1, keepdims=True)) @ x) / (x ** 2).sum()
        mean = y.mean(axis=1)

        res = pd.DataFrame({
            key: weekly.index,
            "window": label,
            "weeks": w,
            "window_start": window_start,
            "total_events": y.sum(axis=1),
            "mean_weekly": mean,
            "slope": slope,
            # slope relative to the average weekly level, so busy and quiet areas compare
            "rel_slope": np.where(mean > 0, slope / np.where(mean > 0, mean, 1), np.nan),
            "first_seen": first_seen.values,
        })
        too_new = res["first_seen"] > window_start
        res.loc[too_new, ["slope", "rel_slope"]] = np.nan
        res["status"] = np.where(too_new, "NA_insufficient_history", "ok")
        out.append(res)

    return pd.concat(out, ignore_index=True)


def run(gdf, sub_event_types=None, as_of=None, windows=WINDOWS):
    """
    Returns (points, governorates):
      points        GeoDataFrame, one row per point per window
      governorates  DataFrame, one row per governorate per window
    """
    import geopandas as gpd

    # Anchor the window end to the full dataset, not the filtered subset, so that
    # filtering to a rare event type doesn't shift the time window.
    if as_of is None:
        as_of = pd.to_datetime(gdf["YEAR_WEEK"].str.split("/").str[0]).max()

    d = prepare(gdf, sub_event_types)

    pts = rate_of_change(d, "point_id", windows, as_of)
    meta = (d.drop_duplicates("point_id")[["point_id", "geometry", "adm2_name"]]
              .reset_index(drop=True))
    pts = pts.merge(pd.DataFrame(meta), on="point_id", how="left")
    pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=gdf.crs)

    gov = rate_of_change(d, "adm2_name", windows, as_of)
    return pts, gov


# All sub-event types
pts_all, gov_all = run(gdf_events_joined)

# Or specific types (names must match your SUB_EVENT_TYPE values)
# pts_sel, gov_sel = run(gdf, sub_event_types=["Attack", "Mob violence"])

# Wide view of governorate slopes by window
print(gov_all.pivot(index="adm2_name", columns="window", values="slope").round(2))

gov_all.to_csv(f"{PROCESSED}/governorate_rate_of_change.csv", index=False)

pts_csv = pts_all.copy()
pts_csv["lon"] = pts_csv.geometry.x
pts_csv["lat"] = pts_csv.geometry.y
pts_csv = pts_csv.drop(columns="geometry")
pts_csv.to_csv(f"{PROCESSED}/point_rate_of_change.csv", index=False)

### Export Results

In [ ]:
# geopackage for further geoprocessing and visualization
gdf_events_joined.to_file(
    PATH_TO_GEOLOCATED_EVENTS_GPKG,
    driver='GPKG'
)

# df for further analysis
df_final = pd.DataFrame(gdf_events_joined.drop(columns='geometry'))
df_final.to_csv(PATH_TO_GEOLOCATED_EVENTS_DF, index=False)